In [0]:
%pip install yfinance

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

In [0]:
bronze_df = spark.table("raj.bronze.stock_prices")
print(bronze_df.count())

display(bronze_df)

In [0]:
print("-----NULL VALUES-----")
for col_name in bronze_df.columns:
    null_count = bronze_df.filter(col(col_name).isNull()).count()
    print(f"{col_name}: {null_count}")

print("-----DUPLICATE ROWS-----")
tot_rows = bronze_df.count()
dis_rows = bronze_df.select("trade_date","ticker").distinct().count()
print(f"Total rows: {tot_rows}")
print(f"Distinct rows: {dis_rows}")
print(f"Duplicate rows: {tot_rows - dis_rows}")

print("-----DATA TYPES-----")
bronze_df.printSchema()

print("-----NEGATIVE VALUES-----")
neg_close = bronze_df.filter(col("close_price")<0).count()
neg_volume = bronze_df.filter(col("volume")<0).count()
print(f"Negative close prices: {neg_close}")
print(f"Negative volume: {neg_volume}")

print("----DATE RANGES----")
display(bronze_df.select(min("trade_date").alias("earliest_date"),max("trade_date").alias("latest_date")))


In [0]:
bronze_df = spark.table("raj.bronze.stock_prices")
bronze_dedupe_df = bronze_df.dropDuplicates(["trade_date","ticker"])

print(bronze_df.count())
print(bronze_dedupe_df.count())
bronze_dedupe_df.printSchema()

bronze_dedupe_df.write.mode("overwrite").format("delta").option("overwriteSchema","true").saveAsTable("raj.bronze.stock_prices")

In [0]:
silver_df = bronze_df.withColumn("trade_date",col("trade_date").cast(DateType())).dropDuplicates(["trade_date","ticker"]).orderBy("ticker","trade_date")

silver_df.printSchema()
silver_df.count()
display(silver_df)

In [0]:
silver_df = silver_df.withColumn("daily_range",round(col("high_price")-col("low_price"),2))
#daily range creation
display(silver_df.select("trade_date","ticker","high_price","low_price","daily_range"))

In [0]:
from pyspark.sql.window import *
#using window function to look at prev day close to get daily return
window_prev_close = Window.partitionBy("ticker").orderBy(col("trade_date"))

silver_df = silver_df.withColumn("prev_close", lag("close_price", 1).over(window_prev_close))

silver_df = silver_df.withColumn("daily_return",round((col("close_price")-col("prev_close"))/col("prev_close")*100,2))

display(silver_df)


In [0]:
from pyspark.sql.window import *
#MOving average for 7ds
window_7d = Window.partitionBy("ticker").orderBy("trade_date").rowsBetween(-6,0)

silver_df = silver_df.withColumn("moving_avg_7d", round(
    avg(col("close_price")).over(window_7d),2)
)

display(silver_df.select(
    "trade_date", 
    "ticker", 
    "close_price", 
    "moving_avg_7d",
    "daily_return"
).limit(10))

In [0]:
from pyspark.sql.window import *
#Moving avg for 30ds
window_30d = Window.partitionBy("ticker").orderBy("trade_date").rowsBetween(-29,0)

silver_df = silver_df.withColumn("moving_avg_30d", round(
    avg(col("close_price")).over(window_30d),2)
)

display(silver_df.select(
    "trade_date", 
    "ticker", 
    "close_price", 
    "moving_avg_7d",
    "moving_avg_30d",
    "daily_return"
).limit(10))

In [0]:
from datetime import datetime
from pyspark.sql.functions import lit, when
print(silver_df.columns)
# Add silver layer metadata
silver_df = silver_df \
    .withColumn("silver_timestamp", lit(datetime.now())) \
    .withColumn("is_valid", 
        when(
            (col("close_price").isNotNull()) & 
            (col("close_price") > 0) & 
            (col("volume") >= 0)  # Fixed: volume not colume
        , True).otherwise(False)
    ).drop("prev_close")

display(silver_df.limit(5))

In [0]:
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("raj.silver.stock_prices")